# Quickstart Notebook (parametrizável)

Este notebook demonstra um fluxo end-to-end usando o pipeline do projeto:
- ingestão de dados de um arquivo CSV de exemplo
- processamento da base canônica
- cálculo de retornos diários
- salvamento de artefatos (CSV + gráfico)

Ele foi projetado para ser executado **headlessly** via **papermill** usando parâmetros.

## Parâmetros (papermill)

Para executar de forma parametrizada, passe valores para estas variáveis via `papermill`.

In [7]:
# Parameters
# These defaults are used when running interactively. Papermill can override them.
# Example:
#   papermill quickstart.ipynb quickstart-output.ipynb \
#     -p tickers '["PETR4"]' \
#     -p output_dir outputs/notebooks/quickstart

# Parameter for a list of tickers. The notebook will process the first ticker.
tickers = ["PETR4"]

# Backwards compatibility (a later cell derives `ticker` from `tickers`).
# Do not reassign `ticker` here so papermill can override `tickers` correctly.

start_date = None  # string YYYY-MM-DD or None
end_date = None  # string YYYY-MM-DD or None
output_dir = "outputs/notebooks/quickstart"
csv_path = "dados/ticker_example.csv"  # path to example dataset

In [ ]:
import os
import sys
from pathlib import Path
from typing import cast

# Ensure we can locate the repository root even when the notebook runs in CI from an arbitrary cwd.

def find_repo_root(start: Path | None = None) -> Path:
    if start is None:
        start = Path.cwd()
    for p in (start, *start.parents):
        if (p / "pyproject.toml").exists() or (p / ".git").exists():
            return p
    raise FileNotFoundError("Could not locate repository root (pyproject.toml or .git missing)")

repo_root = find_repo_root()

# Ensure project root is on sys.path so imports like `src.*` and `scripts.*` work.
sys.path.insert(0, str(repo_root))

# Ensure output directory exists
output_dir = (repo_root / output_dir).expanduser()
output_dir.mkdir(parents=True, exist_ok=True)

# Guard against missing ticker values and verify proper input shape.
if tickers is None:
    raise ValueError("Parâmetro 'tickers' não pode ser None. Forneça pelo menos um ticker.")
if not isinstance(tickers, (list, tuple)):
    raise TypeError("Parâmetro 'tickers' deve ser uma lista ou tupla de strings.")
if not tickers:
    raise ValueError("Parâmetro 'tickers' não pode ser vazio. Forneça pelo menos um ticker.")

# A partir daqui, `tickers` é list/tuple não vazio.
assert isinstance(tickers, (list, tuple)) and tickers, "tickers deve ser lista/tupla não vazia"

# type narrowing for static checkers (Pylance reportOptionalSubscript)
# O cast é seguro porque validamos `tickers` acima.
ticker = cast(str, tickers[0])

# Ensure the CSV adapter uses the correct file when running from CI/automation.
os.environ["CSV_ADAPTER_FILE"] = str((repo_root / csv_path).expanduser())

print(f"Repository root: {repo_root}")
print(f"Using ticker: {ticker}")
print(f"Output directory: {output_dir}")
print(f"CSV data file: {os.environ.get('CSV_ADAPTER_FILE')}")


Repository root: /home/phbr/Projects/01-owned/Analise-financeira-B3
Using ticker: PETR4
Output directory: /home/phbr/Projects/01-owned/Analise-financeira-B3/outputs/notebooks/quickstart
CSV data file: /home/phbr/Projects/01-owned/Analise-financeira-B3/dados/ticker_example.csv


In [9]:
# Ensure the database schema exists (idempotent).
# This is safe to run repeatedly and ensures the pipeline has a DB to write to.

from scripts.apply_migrations import main as _apply_migrations

_apply_migrations()

{"time": "2026-03-20T00:49:21.519082Z", "level": "INFO", "logger": "scripts.apply_migrations", "message": "migrations applied to ./dados/data.db", "taskName": "Task-51"}


In [10]:
from src.ingest.pipeline import ingest
from src.retorno import compute_returns

# Run the core ETL pipeline (ingest -> snapshot persistence)
ingest_result = ingest(
    ticker=ticker,
    source="csv",
    dry_run=False,
    force_refresh=True,
    start=start_date,
    end=end_date,
)

print("Ingest result:", ingest_result)

# Compute returns and persist to the database (returns table)
returns_df = compute_returns(
    ticker,
    start=start_date,
    end=end_date,
    dry_run=False,
)

# Sanity check (will raise if expectations are violated)
assert returns_df is not None and not returns_df.empty, "Retornos não foram calculados"
assert "return_value" in returns_df.columns, "Coluna 'return_value' ausente"

# Save results for inspection
returns_csv = output_dir / f"{ticker}_returns.csv"
returns_df.to_csv(returns_csv, index=False)
print(f"Saved returns to {returns_csv}")

{"time": "2026-03-20T00:49:21.538877Z", "level": "INFO", "logger": "src.ingest.pipeline", "message": "pipeline.ingest start", "job_id": "de6635ec-c315-4e7a-b365-b089b180885c", "source": "csv", "started_at": "2026-03-20T00:49:21.538852Z", "taskName": "Task-54", "ticker": "PETR4", "force_refresh": true, "dry_run": false}
{"time": "2026-03-20T00:49:21.555130Z", "level": "INFO", "logger": "src.etl.mapper", "message": "Mapped 2 rows for PETR4 from csv (checksum: 7b935e3b...)", "taskName": "Task-54"}
{"time": "2026-03-20T00:49:21.571396Z", "level": "INFO", "logger": "src.ingest.snapshot_ingest", "message": "force refresh requested for PETR4; bypassing snapshot cache", "taskName": "Task-54"}
{"time": "2026-03-20T00:49:21.606496Z", "level": "INFO", "logger": "src.ingest.snapshot_ingest", "message": "ingest_from_snapshot complete for PETR4: rows_processed=2, cached=False, elapsed=0.04s", "taskName": "Task-54"}
{"time": "2026-03-20T00:49:21.615017Z", "level": "INFO", "logger": "src.ingest.pipeli

In [11]:
# Optional: Basic visualization (saved to file, no display required)

try:
    import matplotlib
    matplotlib.use("Agg")
    import matplotlib.pyplot as plt

    fig, ax = plt.subplots(figsize=(8, 4))
    ax.plot(returns_df["date"], returns_df["return_value"].cumsum(), marker="o")
    ax.set_title(f"Retorno acumulado - {ticker}")
    ax.set_xlabel("Data")
    ax.set_ylabel("Retorno acumulado")
    ax.grid(True)

    plot_path = output_dir / f"{ticker}_cumulative_return.png"
    fig.tight_layout()
    fig.savefig(plot_path, dpi=150)
    print(f"Saved plot to {plot_path}")
except ImportError:
    print("matplotlib not installed; skipping plot generation")

Saved plot to /home/phbr/Projects/01-owned/Analise-financeira-B3/outputs/notebooks/quickstart/PETR4_cumulative_return.png
